# Catalog-Linked Database (CLD) Lab Guide

This notebook walks you through creating a **Catalog-Linked Database** in Snowflake that connects to an existing AWS Glue Iceberg REST catalog.

A Catalog-Linked Database auto-discovers and syncs schemas and tables from your external catalog — no data copy, no ETL pipeline required.

**What you'll do:**
1. Create a Catalog Integration (connection to Glue)
2. Create a Catalog-Linked Database (auto-syncs schemas/tables)
3. Verify the link and discover tables
4. Query Iceberg data directly through Snowflake

---

## Configure Your Environment

Set the variables below to match your environment. All subsequent SQL cells reference these variables via Jinja templating.

In [ ]:
SF_ROLE = "ACCOUNTADMIN"
CATALOG_INTEGRATION_NAME = "glue_rest_catalog_int"
CLD_DATABASE = "balloon_game_events"
GLUE_NAMESPACE = "balloon_pops"
GLUE_TABLE = "balloon_game_events"

---

## Step 1: Set Role

In [ ]:
%%sql
USE ROLE {{SF_ROLE}};

---

## Step 2: Create Catalog Integration

A **Catalog Integration** tells Snowflake how to connect to your external catalog (in this case, AWS Glue via the Iceberg REST protocol).

Key settings:
- **ICEBERG_REST** catalog source — connects via the Iceberg REST API
- **SIGV4** authentication — Snowflake signs requests using your AWS role
- **VENDED_CREDENTIALS** — the catalog issues short-lived S3 credentials (no external volume needed)

Replace the `<PLACEHOLDER>` values with your Glue catalog ARN, role ARN, and region.

In [ ]:
%%sql
CREATE OR REPLACE CATALOG INTEGRATION {{CATALOG_INTEGRATION_NAME}}
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  CATALOG_NAMESPACE = '{{GLUE_NAMESPACE}}'
  REST_CONFIG = (
    CATALOG_URI = 'https://glue.<REGION>.amazonaws.com/iceberg'
    CATALOG_API_TYPE = AWS_GLUE
    WAREHOUSE = '<CATALOG_NAME>'
  )
  REST_AUTHENTICATION = (
    TYPE = SIGV4
    SIGV4_IAM_ROLE = '<SIGV4_IAM_ROLE>'
    SIGV4_SIGNING_REGION = '<REGION>'
  )
  ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  ENABLED = TRUE;

---

## Step 3: Create Catalog-Linked Database

A **Catalog-Linked Database** automatically discovers and syncs schemas and Iceberg tables from the external catalog into Snowflake. No manual table registration required.

In [ ]:
%%sql
CREATE OR REPLACE DATABASE {{CLD_DATABASE}}
  LINKED_CATALOG = (
    CATALOG_INTEGRATION = '{{CATALOG_INTEGRATION_NAME}}'
  )
  COMMENT = 'Glue Iceberg CLD with vended credentials';

---

## Step 4: Verify Link Status

Check that the catalog link is active and syncing. Expect `"executionState": "RUNNING"` with no failure details.

In [ ]:
%%sql
SELECT SYSTEM$CATALOG_LINK_STATUS('{{CLD_DATABASE}}');

---

## Step 5: Discover Schemas and Tables

Once the link is active, Snowflake auto-discovers schemas and Iceberg tables from the Glue catalog. List them below.

In [ ]:
%%sql
SHOW SCHEMAS IN DATABASE {{CLD_DATABASE}};
SHOW ICEBERG TABLES IN SCHEMA {{CLD_DATABASE}}."{{GLUE_NAMESPACE}}";

---

## Step 6: Query Bronze Data

Query the Iceberg table data directly through Snowflake. The `event` column contains JSON — use `PARSE_JSON` to extract individual fields.

In [ ]:
%%sql
SELECT
  PARSE_JSON(event):player::STRING AS player,
  PARSE_JSON(event):balloon_color::STRING AS balloon_color,
  PARSE_JSON(event):score::INTEGER AS score,
  PARSE_JSON(event):event_ts::TIMESTAMP_TZ AS event_ts
FROM {{CLD_DATABASE}}."{{GLUE_NAMESPACE}}"."{{GLUE_TABLE}}"
LIMIT 10;

---

## What Just Happened?

You now have Snowflake reading Iceberg tables **directly from AWS Glue** — no data copy, no ETL pipeline.

| What | How |
|---|---|
| **Catalog Integration** | Snowflake connects to Glue via the Iceberg REST API using SigV4 authentication |
| **Vended Credentials** | The catalog issues short-lived S3 credentials — no external volume needed |
| **Catalog-Linked Database** | Glue namespaces and tables auto-sync as Snowflake schemas and Iceberg tables |
| **Live Queries** | `SELECT` reads the latest committed Iceberg snapshots in real time |

The CLD stays in sync automatically. When new tables or schemas appear in Glue, Snowflake discovers them without any manual refresh.

**Next up:** Build Dynamic Iceberg Tables to transform this raw bronze JSON into production-ready silver aggregates.

In [ ]:
%%sql
DROP DATABASE IF EXISTS {{CLD_DATABASE}};
DROP CATALOG INTEGRATION IF EXISTS {{CATALOG_INTEGRATION_NAME}};